In [ ]:
! pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 15.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [3]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

In [4]:
CHECKPOINT = "google-t5/t5-small"
DATASET = "Helsinki-NLP/opus_books"

In [5]:
t5_tokenizer = T5Tokenizer.from_pretrained(CHECKPOINT)
t5_model = T5ForConditionalGeneration.from_pretrained(CHECKPOINT)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
# load dataset
dataset = load_dataset(DATASET, "en-fr")['train']

# shuffle the dataset
dataset = dataset.shuffle(seed=42)

# Split the dataset into train, valid, test(train_size=80%, valid_size=10%, test_size=10%)
train_valid_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.9, 0.1])
train_dataset, valid_dataset = torch.utils.data.random_split(train_valid_dataset, [0.9, 0.1])

test_dataset, _ = torch.utils.data.random_split(test_dataset, [0.2, 0.8]) # Further reduce the size since otherwise computations are slow

README.md:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

#### sample code to give you an idea of how the T5 toknizer works (make sure to remove this cell when you want to push your code)

In [7]:
dataset[0]

{'id': '44126',
 'translation': {'en': '"Not if it were my own brother!" cried d’Artagnan, as if carried away by his enthusiasm.',
  'fr': "«Non, fût-ce mon frère!» s'écria d'Artagnan comme emporté par l'enthousiasme."}}

In [8]:
example = dataset[0]
text = example['translation']['en']
text_target = example['translation']['fr']

encoded = t5_tokenizer(
    text=text,
    text_target=text_target,
    padding=True,
    truncation=True,
    return_tensors="pt",
)

input_ids = encoded["input_ids"]
labels = encoded["labels"]
attention_mask = encoded["attention_mask"]
decoder_input_ids = t5_model._shift_right(labels)

In [9]:
# Create custom Dataset class
class TranslationDataset(Dataset):
    def __init__(
        self,
        data,
        tokenizer: T5Tokenizer,
        max_length: int = 128,
        source_lang: str = "en",
        target_lang: str = "fr",
        task_prefix: str = "translate English to French: "
    ):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.source_lang = source_lang
        self.target_lang = target_lang
        self.task_prefix = task_prefix

    def __len__(self):
        return len(self.data)


    def __getitem__(self, idx):
        """
        Get a single example from the dataset.

        Args:
            idx: Index of the example

        Returns:
            Dictionary containing input_ids, decoder_input_ids, labels, attention_mask
        """
        example = self.data[idx]
        source_text = self.task_prefix + example['translation'][self.source_lang]
        target_text = example['translation'][self.target_lang]

        # Tokenize inputs and targets
        encoded = self.tokenizer(
            text=source_text,
            text_target=target_text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        features = {
            'input_ids': encoded["input_ids"].squeeze(0),
            'decoder_input_ids': encoded["input_ids"].squeeze(0),
            'attention_mask': encoded["attention_mask"].squeeze(0),
            'labels': encoded["labels"].squeeze(0),
        }
        return features

        # 1. prepend the task_prefix to english sentence and generate source_text
        # 2. generate target_text from french sentence
        # 3. pass the source_text and target_text to the toknizer
        # 4. build the features dictionary and return it

        # features =  {
        #     'input_ids': TODO,
        #     'attention_mask': TODO,
        #     'labels': TODO,
        #     'decoder_input_ids': TODO
        # }
        # return features

### Create DataLoader

In [10]:
def create_dataloader(dataset, tokenizer, batch_size, max_length):
    """
    Create a DataLoader for a given dataset
    """
    translation_dataset = TranslationDataset(dataset, tokenizer, max_length)

    return DataLoader(
        translation_dataset,
        batch_size=batch_size,
    )

In [11]:
# prepare dataloaders for train, valid, test dataset
BATCH_SIZE = 16
MAX_LENGTH = 128

train_loader = create_dataloader(train_dataset, t5_tokenizer, BATCH_SIZE, MAX_LENGTH)
valid_loader = create_dataloader(valid_dataset, t5_tokenizer, BATCH_SIZE, MAX_LENGTH)
test_loader = create_dataloader(test_dataset, t5_tokenizer, BATCH_SIZE, MAX_LENGTH)

In [14]:
next(iter(train_loader))

{'input_ids': tensor([[13959,  1566,    12,  ...,     0,     0,     0],
         [13959,  1566,    12,  ...,     0,     0,     0],
         [13959,  1566,    12,  ...,     0,     0,     0],
         ...,
         [13959,  1566,    12,  ...,     0,     0,     0],
         [13959,  1566,    12,  ...,     0,     0,     0],
         [13959,  1566,    12,  ...,     0,     0,     0]]),
 'decoder_input_ids': tensor([[13959,  1566,    12,  ...,     0,     0,     0],
         [13959,  1566,    12,  ...,     0,     0,     0],
         [13959,  1566,    12,  ...,     0,     0,     0],
         ...,
         [13959,  1566,    12,  ...,     0,     0,     0],
         [13959,  1566,    12,  ...,     0,     0,     0],
         [13959,  1566,    12,  ...,     0,     0,     0]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1, 

In [1]:
import torch
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration
from data_preprocessing import create_dataloader
from configs import CHECKPOINT, DATASET, BATCH_SIZE, MAX_LENGTH

t5_tokenizer = T5Tokenizer.from_pretrained(CHECKPOINT)
t5_model = T5ForConditionalGeneration.from_pretrained(CHECKPOINT)

# Load dataset
dataset = load_dataset(DATASET, "en-fr")['train']

# Shuffle the dataset
dataset = dataset.shuffle(seed=42)

# Split the dataset into train, valid, test (train_size=80%, valid_size=10%, test_size=10%)
train_valid_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.9, 0.1])
train_dataset, valid_dataset = torch.utils.data.random_split(train_valid_dataset, [0.9, 0.1])



/Users/zeinab/mambaforge/envs/llm_prefix_tuning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [3]:
# Prepare dataloaders for train, valid, and test datasets
train_loader = create_dataloader(
    dataset=train_dataset, 
    tokenizer=t5_tokenizer, 
    shift_right_func=t5_model._shift_right,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

valid_loader = create_dataloader(
    dataset=valid_dataset, 
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

test_loader = create_dataloader(
    dataset=test_dataset, 
    tokenizer=t5_tokenizer, 
    shift_right_func=t5_model._shift_right, 
    batch_size=BATCH_SIZE, 
    max_length=MAX_LENGTH
)


In [4]:
print(f"Number of batches in train: {len(train_loader)}")
print(f"Number of batches in valid: {len(valid_loader)}")
print(f"Number of batches in test: {len(test_loader)}")

Number of batches in train: 6434
Number of batches in valid: 715
Number of batches in test: 795


In [5]:
# Get a sample batch
sample_batch = next(iter(train_loader))
    
print("\nSample batch shapes:")
for key, value in sample_batch.items():
    print(f"{key}: {value.shape}")

# Decode a sample from the batch
print("\nSample from batch:")
print("Input:", t5_tokenizer.decode(sample_batch['input_ids'][0].tolist(), skip_special_tokens=True))
print("Target:", t5_tokenizer.decode(sample_batch['labels'][0].tolist(), skip_special_tokens=True))


Sample batch shapes:
input_ids: torch.Size([16, 128])
attention_mask: torch.Size([16, 128])
labels: torch.Size([16, 128])
decoder_input_ids: torch.Size([16, 128])

Sample from batch:
Input: translate English to French: "And men take care that they should."
Target: – Et les hommes font bien ce qu’ils peuvent pour le leur faire croire.
